In [3]:
import yfinance as yf
import pandas as pd
import numpy as np

import requests
from io import StringIO

headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"}
html = requests.get(
    "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
    headers=headers
).text

sp500_table = pd.read_html(StringIO(html))[0]
sp500_table = sp500_table[["Symbol", "Security", "GICS Sector", "GICS Sub-Industry", "Date added"]]
sp500_table.columns = ["ticker", "company", "gics_sector", "gics_sub_industry", "date_added"]
sp500_table["ticker"] = sp500_table["ticker"].str.replace(".", "-", regex=False)

tickers = sp500_table["ticker"].tolist()

# ── 2. DOWNLOAD OHLCV FOR ALL TICKERS + S&P 500 INDEX ─────────────────────────
START = "2019-01-01"
END   = "2024-12-31"

# Download all tickers in one call (much faster than looping)
# auto_adjust=True gives adjusted close prices built-in
raw = yf.download(
    tickers + ["^GSPC"],
    start=START,
    end=END,
    auto_adjust=True,
    progress=True,
)
# raw is a MultiIndex DataFrame: (price_type, ticker)

# ── 3. EXTRACT ADJUSTED CLOSE FOR ALL STOCKS ──────────────────────────────────
close = raw["Close"].copy()          # shape: (trading_days, 504)
volume = raw["Volume"].copy()

# Separate market benchmark from individual stocks
market_close = close.pop("^GSPC")   # Series for S&P 500 index
# close now has only the 503 stock tickers

# ── 4. COMPUTE DAILY RETURNS ───────────────────────────────────────────────────
raw_returns    = close.pct_change(fill_method=None)
market_returns = market_close.pct_change(fill_method=None).rename("market")

# Market-adjusted (excess) returns: remove the common market tide
excess_returns = raw_returns.subtract(market_returns, axis=0)

# ── 5. MELT TO LONG FORMAT (one row per stock-day) ────────────────────────────
def melt_to_long(df, value_name):
    return (df
        .reset_index()
        .melt(id_vars="Date", var_name="ticker", value_name=value_name)
    )

df_raw     = melt_to_long(raw_returns,    "raw_daily_return")
df_excess  = melt_to_long(excess_returns, "excess_daily_return")
df_close   = melt_to_long(close,          "adj_close")
df_volume  = melt_to_long(volume,         "volume")
df_market  = market_returns.reset_index().rename(columns={"^GSPC": "market_daily_return"})

# Join all on Date + ticker
df = (df_close
    .merge(df_volume,   on=["Date", "ticker"], how="left")
    .merge(df_raw,      on=["Date", "ticker"], how="left")
    .merge(df_excess,   on=["Date", "ticker"], how="left")
    .merge(df_market,   on="Date",             how="left")
    .merge(sp500_table, on="ticker",           how="left")
)
df = df.sort_values(["ticker", "Date"]).reset_index(drop=True)

# ── 6. FORWARD 5-DAY RETURN + DIRECTION LABEL (target variable) ───────────────
df["forward_5day_return"] = (
    df.groupby("ticker")["adj_close"]
      .transform(lambda x: x.shift(-5) / x - 1)
)
df["forward_5day_direction"] = (df["forward_5day_return"] > 0).astype(int)

# ── 7. DROP ROWS WITH NO PRICE DATA (tickers added mid-period, etc.) ──────────
df = df.dropna(subset=["adj_close", "raw_daily_return"])

print(f"Rows:    {len(df):,}")
print(f"Tickers: {df['ticker'].nunique()}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")
print(df.head())
print(df.dtypes)

[********              16%                       ]  81 of 504 completed$SNDK: possibly delisted; no price data found  (1d 2019-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1546318800, endDate = 1735621200")
[**********************86%****************       ]  434 of 504 completed$Q: possibly delisted; no price data found  (1d 2019-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1546318800, endDate = 1735621200")
[*********************100%***********************]  504 of 504 completed

2 Failed downloads:
['SNDK', 'Q']: possibly delisted; no price data found  (1d 2019-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1546318800, endDate = 1735621200")


Rows:    743,837
Tickers: 501
Date range: 2019-01-03 → 2024-12-30
        Date ticker  adj_close     volume  raw_daily_return  \
1 2019-01-03      A  60.005020  5383900.0         -0.036840   
2 2019-01-04      A  62.082012  3123700.0          0.034614   
3 2019-01-07      A  63.400265  3235100.0          0.021234   
4 2019-01-08      A  64.329704  1578100.0          0.014660   
5 2019-01-09      A  65.676437  2442300.0          0.020935   

   excess_daily_return    market               company  gics_sector  \
1            -0.012083 -0.024757  Agilent Technologies  Health Care   
2             0.000278  0.034336  Agilent Technologies  Health Care   
3             0.014224  0.007010  Agilent Technologies  Health Care   
4             0.004965  0.009695  Agilent Technologies  Health Care   
5             0.016837  0.004098  Agilent Technologies  Health Care   

                gics_sub_industry  date_added  forward_5day_return  \
1  Life Sciences Tools & Services  2000-06-05             